# Neural Network for Stress Prediction

Surrogate model to predict stress from wing geometry parameters (`W1`, `W2`, `R`, `t`), trained on FEA simulation data. Architecture and hyperparameters are selected via cross-validation and compared against the GPR baseline.

## Imports

In [1]:
import time
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

In [2]:
# Set a fixed random state for reproducibility across all random operations
random_state = 42

# Set the path to the dataset
data_path = "../data/TrainingData.csv"

In [3]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


## Data Pre-Processing

### Load & Inspect Dataset

In [4]:
# Load the dataset, including missing headers and assigning column names
df = pd.read_csv(data_path, header=None, names=['W1', 'W2', 'R', 't', 'stress'])

### Train / Test Split

Hold out 20% as a test set — never touched during architecture search or training.

In [5]:
# split into features and target
X = df.drop('stress', axis=1)
y = df['stress']

# split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

In [6]:
# Scale the features and target variable
def scale_data(X_train, X_test, y_train, y_test):
    # Scale the features using StandardScaler
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    # Scale the target variable using StandardScaler
    y_scaler = StandardScaler()
    y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1)).flatten()
    y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1)).flatten()
    return X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, y_scaler

X_train_scaled, X_test_scaled, y_train_scaled, y_test_scaled, y_scaler = scale_data(X_train, X_test, y_train, y_test)

## Model Architecture & Hyperparameters

### Model Setup

Build DataLoaders, define activation lookup, and construct the `StressNet` module used by all experiments.

In [7]:
# create PyTorch datasets and dataloaders
train_dataset = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32), torch.tensor(y_train_scaled, dtype=torch.float32))
test_dataset = TensorDataset(torch.tensor(X_test_scaled, dtype=torch.float32), torch.tensor(y_test_scaled, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

# Inspect the shape of the first batch of data
for X, y in test_loader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([10, 4])
Shape of y: torch.Size([10]) torch.float32


In [13]:
# define a dictionary to store depth, width and activation function combinations to assess in the grid search
architectures = {
      'shallow_wide':  {'hidden_dims': [128, 128],        'activation': 'relu',  'dropout':
   0.0},
      'deep_narrow':   {'hidden_dims': [32, 32, 32, 32],  'activation': 'relu',  'dropout':
   0.0},
      'deep_wide':     {'hidden_dims': [64, 64, 64],      'activation': 'relu',  'dropout':
   0.1},
      'tanh_medium':   {'hidden_dims': [64, 64],           'activation': 'tanh',
  'dropout': 0.0},
  }

# define a dictionary to map activation function names to their corresponding PyTorch classes
_ACTIVATIONS = {'relu': nn.ReLU, 'tanh': nn.Tanh, 'elu': nn.ELU}

In [15]:
# Define the StressNet class
class StressNet(nn.Module):
    # Constructor for StressNet, which takes in the number of input features and a configuration dictionary to build the network architecture
    def __init__(self, in_features: int, config: dict):
        super().__init__()
        activation_cls = _ACTIVATIONS[config['activation']] # get the activation class from the dictionary based on the config
        dropout        = config.get('dropout', 0.0) # get the dropout value from the config, defaulting to 0.0 if not specified
        dims           = [in_features] + config['hidden_dims'] + [1] # create a list of layer dimensions based on the input features, hidden layer dimensions from the config, and an output layer with 1 neuron

        # Define empty list to hold the layers of the network
        layers = []
        # Loop through the dimensions to create the hidden layers with the specified activation function and optional dropout
        for in_d, out_d in zip(dims[:-2], dims[1:-1]):
            # Add a linear layer followed by the activation function to the layers list
            layers += [nn.Linear(in_d, out_d), activation_cls()]
            # If dropout is specified in the config, add a dropout layer after the activation function
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
        # Finally, add the output layer without an activation function to the layers list
        layers.append(nn.Linear(dims[-2], dims[-1]))  # output layer — no activation

        # Combine all the layers into a sequential model and assign it to self.net
        self.net = nn.Sequential(*layers)

    # define the forward pass of the network, which takes an input tensor x and returns the predicted stress values
    def forward(self, x):
        return self.net(x).squeeze(-1)

### Baseline Model

1 hidden layer, 32 neurons, ReLU, Adam (lr=0.001), MSE loss, no regularisation. Goodfellow (Ch. 11.2): *"use a feedforward network with fully connected layers… begin by using some kind of piecewise linear unit (ReLUs)… A very reasonable alternative is Adam."*

All subsequent experiments compare against these baseline metrics.

In [12]:
# function to inverse transform the scaled predictions back to the original scale
def unravel_predictions(preds_scaled, y_scaler):
    preds_scaled_reshaped = preds_scaled.reshape(-1, 1)
    preds_unscaled = y_scaler.inverse_transform(preds_scaled_reshaped).flatten()
    return preds_unscaled

In [16]:
# Define the number of epochs for training the baseline model
N_EPOCHS = 500

# Baseline model architecture - 1 hidden layer with 32 neurons, ReLU activation, no dropout
baseline_config = {'hidden_dims': [32], 'activation': 'relu', 'dropout': 0.0}
# Initialise the baseline model, optimizer and loss function
baseline_model  = StressNet(in_features=4, config=baseline_config).to(device)

# Define optimiser and loss function for training the baseline model
optimiser = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)
loss_fn   = nn.MSELoss()

# --- Training ---
t_train_start = time.perf_counter() # start the timer for training
baseline_model.train() # train baseline model
for epoch in range(N_EPOCHS):
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        loss = loss_fn(baseline_model(X_batch), y_batch)
        loss.backward()
        optimiser.step()
train_time = time.perf_counter() - t_train_start

# --- Prediction & timing ---
baseline_model.eval()
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
t_pred_start = time.perf_counter()
with torch.no_grad():
    preds_scaled = baseline_model(X_test_tensor).cpu().numpy()
pred_time_ms = (time.perf_counter() - t_pred_start) / len(X_test_scaled) * 1e3

# --- Inverse-transform to MPa ---
preds_mpa  = unravel_predictions(preds_scaled, y_scaler)
y_test_mpa = y_test.values

# --- Metrics ---
rmse = np.sqrt(np.mean((preds_mpa - y_test_mpa) ** 2))
mae  = mean_absolute_error(y_test_mpa, preds_mpa)
r2   = r2_score(y_test_mpa, preds_mpa)

baseline_results = {'rmse': rmse, 'mae': mae, 'r2': r2,
                    'train_time_s': train_time, 'pred_time_ms': pred_time_ms}

print("Baseline NN — 1 x 32, ReLU, Adam")
print(f"  Test RMSE  : {rmse:.2f} MPa")
print(f"  Test MAE   : {mae:.2f} MPa")
print(f"  Test R²    : {r2:.4f}")
print(f"  Train time : {train_time:.2f} s ({N_EPOCHS} epochs)")
print(f"  Pred time  : {pred_time_ms:.4f} ms/sample")

Baseline NN — 1 x 32, ReLU, Adam
  Test RMSE  : 1.26 MPa
  Test MAE   : 0.95 MPa
  Test R²    : 0.9771
  Train time : 2.10 s (500 epochs)
  Pred time  : 0.7994 ms/sample


Key design decisions here:
  - dims list builds the layer sequence automatically from hidden_dims —
  adding/removing layers is just editing the list in the config, not
  touching the class
  - Output layer has no activation because this is unbounded regression
  (stress in MPa)
  - dropout=0.0 is a no-op by default so the same class handles both
  regularised and unregularised variants
  - _ACTIVATIONS lookup keeps the config serialisable (strings, not
  objects) — useful if you later want to save configs to JSON alongside
  the model

  Given you only have 50 training samples, I'd keep architectures small
  (≤3 layers, ≤64 units) — deep wide nets will overfit badly at that
  scale, which is partly why GPR tends to do well here.

### Architecture Search

Define candidate architectures (depth, width, activation) and compare via K-Fold cross-validation. Selection metric: CV MAPE.

In [19]:
# Test 1,2 and 3 hidden layers with fixed activation and dropout to see if depth or width has a bigger impact on performance and show validation error during training to see if models are overfitting or underfitting.
def train_and_evaluate_model(config, train_loader, test_loader, y_scaler, device, n_epochs=500):
    model = StressNet(in_features=4, config=config).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn   = nn.MSELoss()

    # Training
    t_train_start = time.perf_counter()
    model.train()
    for epoch in range(n_epochs):
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimiser.zero_grad()
            loss = loss_fn(model(X_batch), y_batch)
            loss.backward()
            optimiser.step()
    train_time = time.perf_counter() - t_train_start

    # Prediction
    model.eval()
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    t_pred_start = time.perf_counter()
    with torch.no_grad():
        preds_scaled = model(X_test_tensor).cpu().numpy()
    pred_time_ms = (time.perf_counter() - t_pred_start) / len(X_test_scaled) * 1e3

    # Inverse-transform to MPa
    preds_mpa  = unravel_predictions(preds_scaled, y_scaler)
    y_test_mpa = y_test.values

    # Metrics
    rmse = np.sqrt(np.mean((preds_mpa - y_test_mpa) ** 2))
    mae  = mean_absolute_error(y_test_mpa, preds_mpa)
    r2   = r2_score(y_test_mpa, preds_mpa)

    results = {'rmse': rmse, 'mae': mae, 'r2': r2,
               'train_time_s': train_time, 'pred_time_ms': pred_time_ms}
    
    return results

# Candidate architectures: 1, 2, and 3 hidden layers with varying widths
depth_configs = [
    # 1 hidden layer
    [32],
    [64],
    [128],
    # 2 hidden layers
    [64, 32],
    [128, 64],
    # 3 hidden layers
    [128, 64, 32],
]

depth_results = {}
for hidden_dims in depth_configs:
    config = {'hidden_dims': hidden_dims, 'activation': 'relu', 'dropout': 0.0}
    results = train_and_evaluate_model(config, train_loader, test_loader, y_scaler, device, n_epochs=N_EPOCHS)
    key = str(hidden_dims)
    depth_results[key] = results
    print(f"NN {hidden_dims} — ReLU, Adam")
    print(f"  Test RMSE  : {results['rmse']:.2f} MPa")
    print(f"  Test MAE   : {results['mae']:.2f} MPa")
    print(f"  Test R²    : {results['r2']:.4f}")
    print(f"  Train time : {results['train_time_s']:.2f} s ({N_EPOCHS} epochs)")
    print(f"  Pred time  : {results['pred_time_ms']:.4f} ms/sample")
    print()

NN [32] — ReLU, Adam
  Test RMSE  : 1.54 MPa
  Test MAE   : 1.24 MPa
  Test R²    : 0.9659
  Train time : 2.46 s (500 epochs)
  Pred time  : 0.2564 ms/sample

NN [64] — ReLU, Adam
  Test RMSE  : 0.81 MPa
  Test MAE   : 0.70 MPa
  Test R²    : 0.9906
  Train time : 2.19 s (500 epochs)
  Pred time  : 0.8458 ms/sample

NN [128] — ReLU, Adam
  Test RMSE  : 0.57 MPa
  Test MAE   : 0.44 MPa
  Test R²    : 0.9954
  Train time : 2.24 s (500 epochs)
  Pred time  : 13.6540 ms/sample

NN [64, 32] — ReLU, Adam
  Test RMSE  : 0.59 MPa
  Test MAE   : 0.53 MPa
  Test R²    : 0.9950
  Train time : 2.47 s (500 epochs)
  Pred time  : 0.5473 ms/sample

NN [128, 64] — ReLU, Adam
  Test RMSE  : 0.90 MPa
  Test MAE   : 0.77 MPa
  Test R²    : 0.9884
  Train time : 2.60 s (500 epochs)
  Pred time  : 9.1059 ms/sample

NN [128, 64, 32] — ReLU, Adam
  Test RMSE  : 0.54 MPa
  Test MAE   : 0.43 MPa
  Test R²    : 0.9959
  Train time : 2.79 s (500 epochs)
  Pred time  : 0.0708 ms/sample



### Visualisations

#### Loss Curves

Training and validation loss over epochs for the best architecture. Verify convergence and check for overfitting.

#### Architecture Comparison: Predicted vs True

Scatter plot of predicted vs true stress for each candidate architecture. The dashed red line is the ideal fit.

#### Residual Analysis

Residuals (predicted − true) plotted against true stress and against each input feature to diagnose systematic bias.

#### Prediction Surface (W1 vs W2)

3D surface of predicted stress as `W1` and `W2` vary, with `R` and `t` held at their training-set means. Black points show training observations projected onto the surface.

### Model Selection & Export

Save the best-performing model and its scaler for use in the optimisation script.

## Comparison with GPR Baseline

Side-by-side comparison of the best NN against the best GPR model (Matern 2.5 kernel) on the same held-out test set. Metrics: Test RMSE, MAE, MAPE, training time, prediction time.